# KV260 SRCNN 板端验证
沿用 vadd_hls.ipynb 的 PYNQ 流程，复现 tb_srcnn 和 tb_set14。
板上须已有可工作的 PYNQ 环境。将本文件与 src/weights、test/set5、test/set14 一起上传。
将完整 Vivado 系统 bitstream 和对应 Block Design HWH 放在本文件旁，重命名 srcnn.bit 和 srcnn.hwh。
典型来源：project/project.runs/impl_1/design_1_wrapper.bit；
project/project.gen/sources_1/bd/design_1/hw_handoff/design_1.hwh（以实际生成路径为准）。
不能用 HLS IP 的元数据代替完整系统 HWH。
输入是已准备好的 255×255 灰度数据，除以 255；不额外插值。
当前导出顶层是 srcnn，不支持单独读取 Conv1；tb_conv1 需要单独导出 conv1 IP。


In [ ]:
from pathlib import Path
import time, json, csv
import numpy as np
from pynq import Overlay, allocate
ROOT = Path('.').resolve()
BITFILE = ROOT/'srcnn.bit'
IP_NAME = 'srcnn_0'
MODE = 'float'  # 必须与生成 bitstream 时的 SRCNN_INT8 一致，不能由软件切换硬件
TIMEOUT_S = 120.0
RUN_SET14 = True
RESULTS = ROOT/'board_results'
RESULTS.mkdir(exist_ok=True)
assert MODE in ('float','int8')
assert BITFILE.is_file() and BITFILE.with_suffix('.hwh').is_file(), '需要同名系统 bit/hwh'
overlay = Overlay(str(BITFILE))
print(list(overlay.ip_dict))
ip = getattr(overlay,IP_NAME)
print(ip.register_map)


In [ ]:
# 偏移由当前导出的 xsrcnn_hw.h 得到，先与 HWH 验证。
OFFSETS = dict(input_ftmap=0x10,conv1_weights=0x1c,conv1_biases=0x28,
               conv2_weights=0x34,conv2_biases=0x40,conv3_weights=0x4c,
               conv3_biases=0x58,output_ftmap=0x64)
SHAPES = dict(input_ftmap=(1,255,255),output_ftmap=(1,255,255),
              conv1_weights=(64,1,9,9),conv1_biases=(64,),
              conv2_weights=(32,64,1,1),conv2_biases=(32,),
              conv3_weights=(1,32,5,5),conv3_biases=(1,))
registers = overlay.ip_dict[IP_NAME].get('registers',{})
for name,offset in OFFSETS.items():
    matches = [r for key,r in registers.items() if key==name or key.startswith(name+'_')]
    assert matches and min(int(r['address_offset']) for r in matches)==offset, 'HWH寄存器不匹配: '+name

def read_exact(path,dtype,shape):
    path=Path(path); dtype=np.dtype(dtype); count=int(np.prod(shape))
    assert path.is_file(), '缺少文件: '+str(path)
    assert path.stat().st_size==count*dtype.itemsize, '文件尺寸错误: '+str(path)
    return np.fromfile(path,dtype=dtype).reshape(shape)
def load_image(path):
    return read_exact(path,np.uint8,(1,255,255)).astype(np.float32)/np.float32(255)
weights={name:read_exact(ROOT/'src/weights'/(name+'_3x_flp.bin'),'<f4',shape)
         for name,shape in SHAPES.items() if name.startswith('conv')}
buffers={}
try:
    for name,shape in SHAPES.items():
        buffers[name]=allocate(shape=shape,dtype=np.float32)
except Exception:
    for b in buffers.values(): b.freebuffer()
    raise
for name,data in weights.items():
    buffers[name][:]=data
    buffers[name].flush()
# 外部接口始终 float32；地址低/高32位都写，避免截断。
for name,b in buffers.items():
    a=int(b.physical_address); end=a+b.nbytes
    assert (0<=a and end<=0x80000000) or (0x800000000<=a and end<=0x1000000000), '地址超出当前DDR映射: '+name
    ip.write(OFFSETS[name],a&0xffffffff)
    ip.write(OFFSETS[name]+4,(a>>32)&0xffffffff)
    assert ip.read(OFFSETS[name])==(a&0xffffffff)
    assert ip.read(OFFSETS[name]+4)==((a>>32)&0xffffffff)
    print(name,hex(a),b.nbytes,'bytes')
ip.write(0x04,0)
ip.write(0x08,0)
safe_to_free=True


In [ ]:
def run_srcnn(image):
    global safe_to_free
    assert safe_to_free, '之前超时：先重置硬件，勿释放仍被PL访问的缓冲区'
    ctrl=ip.read(0)
    assert ctrl&4, f'IP不处于idle: CTRL={ctrl:#x}'
    assert image.shape==(1,255,255) and np.isfinite(image).all()
    buffers['input_ftmap'][:]=image
    buffers['input_ftmap'].flush()
    buffers['output_ftmap'][:]=np.nan
    buffers['output_ftmap'].flush()
    safe_to_free=False
    start=time.perf_counter(); next_message=start+5
    ip.write(0,1)  # AP_START=1, auto_restart=0
    while True:
        ctrl=ip.read(0)  # AP_DONE清读，仅在这里读取
        now=time.perf_counter()
        if ctrl&2:
            elapsed=now-start
            safe_to_free=True
            break
        if now-start>TIMEOUT_S:
            raise TimeoutError(f'等待{now-start:.1f}s, CTRL={ctrl:#x}; 检查时钟/复位/DDR地址。保留缓冲区直到硬件重置。')
        if now>=next_message:
            print(f'等待{now-start:.1f}s, CTRL={ctrl:#x}；这不是硬件百分比进度')
            next_message=now+5
        time.sleep(0.001)
    buffers['output_ftmap'].invalidate()
    result=np.array(buffers['output_ftmap'],copy=True)
    assert np.isfinite(result).all(), '输出含NaN/Inf，可能未全部写入'
    return result,elapsed
def mse(a,b):
    d=np.asarray(a,dtype=np.float64)-np.asarray(b,dtype=np.float64)
    return float(np.mean(d*d))
def psnr(a,b):
    e=mse(np.clip(a,0,1),np.clip(b,0,1))
    return float('inf') if e==0 else float(-10*np.log10(e))
def tb_psnr(a,b):
    # 对照 util.cpp 的 uint8 转换，模拟截断和回绕；同时报告截幅浮点PSNR。
    a8=(np.trunc(np.asarray(a,dtype=np.float32)*255).astype(np.int64)%256).astype(np.float64)
    b8=(np.trunc(np.asarray(b,dtype=np.float32)*255).astype(np.int64)%256).astype(np.float64)
    e=float(np.mean((a8-b8)**2))
    return float('inf') if e==0 else float(10*np.log10(255**2/e))
rows=[]
lr=load_image(ROOT/'test/set5/butterfly_3x_LR_u8.bin')
golden=read_exact(ROOT/'test/set5/butterfly_3x_GR_flp.bin','<f4',(1,255,255))
hr,seconds=run_srcnn(lr)
golden_error=mse(golden,hr)
max_error=float(np.max(np.abs(golden.astype(np.float64)-hr)))
print('Butterfly: seconds=',seconds,'MSE=',golden_error,'maxerr=',max_error)
rows.append(dict(image='butterfly',seconds=seconds,golden_mse=golden_error,maxerr=max_error))
hr.astype('<f4').tofile(RESULTS/'butterfly_output_flp.bin')
assert golden_error<(1e-10 if MODE=='float' else 3e-4), 'Golden检查失败：检查MODE、参数、缓存和地址'


重复调用及参数更新检查。INT8 对 float 的误差是量化差异；MODE 不改变硬件。

In [ ]:
again,elapsed=run_srcnn(lr)
assert np.array_equal(again,hr), '重复调用输出不同'
print('Repeat PASS:',elapsed,'s')
if MODE=='float':
    delta=np.float32(-0.03125)
    buffers['conv3_biases'][:]=weights['conv3_biases']+delta
    buffers['conv3_biases'].flush()
    try:
        shifted,_=run_srcnn(lr)
        error=float(np.max(np.abs(shifted.astype(np.float64)-hr.astype(np.float64)-float(delta))))
        assert error<1e-5, f'偏置更新失败: {error}'
        print('Runtime bias update PASS:',error)
    finally:
        if safe_to_free:
            buffers['conv3_biases'][:]=weights['conv3_biases']
            buffers['conv3_biases'].flush()


In [ ]:
if RUN_SET14:
    files=sorted((ROOT/'test/set14').glob('*_3x_LR_u8.bin'))
    assert files, '未找到Set14数据'
    for path in files:
        image=load_image(path)
        gt=load_image(path.with_name(path.name.replace('_LR_','_GT_')))
        output,elapsed=run_srcnn(image)
        name=path.name.split('_3x_')[0]
        row=dict(image=name,seconds=elapsed,mse_gt_hr=mse(gt,output),
                 psnr_gt_hr=psnr(gt,output),psnr_gt_lr=psnr(gt,image),
                 tb_psnr_gt_hr=tb_psnr(gt,output),tb_psnr_gt_lr=tb_psnr(gt,image))
        rows.append(row)
        output.astype('<f4').tofile(RESULTS/(name+'_output_flp.bin'))
        print(name,row)
    print('实际图数:',len(files),'；仓库提供13张，不是完整Set14')
fields=sorted({k for row in rows for k in row})
with (RESULTS/'metrics.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=fields); writer.writeheader(); writer.writerows(rows)
with (RESULTS/'run_info.json').open('w') as f:
    json.dump(dict(mode=MODE,bitfile=str(BITFILE),ip=IP_NAME,
                   ip_base=hex(overlay.ip_dict[IP_NAME]['phys_addr']),
                   timing_scope='AP_START至AP_DONE主机轮询耗时，含轮询开销，不含数据加载/分配/缓存同步',
                   results=rows),f,indent=2)
print('结果目录:',RESULTS)


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,data,title in zip(axes,[lr,hr,golden],['LR input','FPGA SRCNN','Golden SRCNN']):
    ax.imshow(np.clip(data[0],0,1),cmap='gray',vmin=0,vmax=1)
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# 超时后勿释放缓冲区。先重置硬件，确认PL停止访问。
if safe_to_free:
    for b in buffers.values(): b.freebuffer()
    buffers.clear()
    print('Buffers released；重跑请从分配单元开始')
else:
    print('调用未完成，保留缓冲区直到硬件重置')


耗时是板端主机测量，不是纯周期计数。Golden MSE检查功能；GT MSE/PSNR检查图像质量。
参考 PYNQ 文档：
https://pynq.readthedocs.io/en/latest/pynq_libraries/allocate.html
https://pynq.readthedocs.io/en/latest/pynq_libraries/overlay.html
本 notebook 已做离线语法检查，尚未在实体板运行。
